<a href="https://colab.research.google.com/github/aiaanad/metopt26/blob/main/lab3/%D0%A2%D0%B5%D0%BC%D0%B0_3_%D0%94%D0%B0%D0%BD%D0%B8%D0%BB%D0%BE%D0%B2%D0%B0_%D0%90%D0%B9%D0%B0%D0%B0%D0%BD%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [81]:
input = {
    'n': 3,
    'f': 'x + 2*y + z',
    'm': 2,
    'g': ['x = y', 'x**2 + y**2 + z**2 = 22']
}

In [75]:
input = {
    'n': 3,
    'f': 'x + y + z',
    'm': 2,
    'g': ['x + y = 0', 'x**2 + y**2 + z**2 = 3']
}

In [82]:
import itertools
import numpy as np
import sympy as sp

def parse_data(data: dict):
  sym_naming = ['x', 'y', 'z']
  n: int = data['n']
  f_expr = sp.sympify(data['f'])

  m = data['m']
  if len(data['g']) != m:
    print("[ERROR]: Incorrect count of limiting functions")
    return

  g_exprs = []
  for i in range(m):
    left, right = data['g'][i].split('=')
    expr_left = sp.sympify(left)
    expr_right = sp.sympify(right)

    eq = sp.Eq(expr_left, expr_right)
    eq = eq.lhs - eq.rhs

    g_exprs.append(eq)

  return n, f_expr, m, g_exprs


def find_nonreg_solves(n: int, f_expr: sp.sympify, m: int, g_exprs: list[sp.sympify]):
  sym_naming = ['x', 'y', 'z']
  syms = sp.symbols(sym_naming[:n])
  # матрица со всеми частными производными длины m
  J = sp.Matrix(g_exprs).jacobian(syms)

  # условие линейной зависимости - все миноры m x m равны нулю
  minor_exprs = []
  if m < n:
    for cols in itertools.combinations(range(n), m):
      minor_exprs.append(J[:, cols].det())
  else:
    minor_exprs.append(sp.S(0))

  system = g_exprs + minor_exprs
  nonreg_points = sp.solve(system, syms, dict=True)

  results = []
  for point in nonreg_points:
    f_val = f_expr.subs(point)
    results.append({
        'point': point,
        'f_val': f_val,
        'type': 'non-regular'
    })

  return results


def find_reg_solves(n:int, f_expr:sp.sympify, m: int, g_exprs: list[sp.sympify]):
  L = f_expr
  lambdas_naming = ['k1', 'k2', 'k3']
  sym_naming = ['x', 'y', 'z']
  syms = sp.symbols(sym_naming[:n])

  lambdas = sp.symbols(lambdas_naming[:m])
  for i in range(m):
    L += g_exprs[i] * lambdas[i]

  diffs = []
  for i in syms + lambdas:
    diffs.append(sp.diff(L, i))

  reg_points = sp.solve(diffs, syms + lambdas, dict=True)
  results = []
  for point in reg_points:
    f_val = f_expr.subs(point)
    results.append({
        'point': point,
        'f_val': f_val,
        'type': 'regular'
    })

  return results


def recognize_type(results: list[dict]):
    valid_points = []
    for p in results:
        try:
            val = float(p['f_val'])
            valid_points.append({'point': p['point'], 'f_val': val, 'type': p['type']})
        except TypeError:
            pass

    if not valid_points:
        print("[-] Действительных точек не найдено.")
        return

    valid_points.sort(key=lambda item: item['f_val'], reverse=True)

    global_max = valid_points[0]
    global_min = valid_points[-1]

    # оставляем только те, строковое представление которых не начинается на 'k'
    def format_coords(point_dict):
        filtered = [f"{k} = {v}" for k, v in point_dict.items() if not str(k).startswith('k')]
        return ", ".join(filtered)

    print(f"[МАКСИМУМ] f = {global_max['f_val']}")
    print(f"   Координаты: {format_coords(global_max['point'])}")
    print(f"   Природа:    {global_max['type']} точка\n")

    print(f"[МИНИМУМ]  f = {global_min['f_val']}")
    print(f"   Координаты: {format_coords(global_min['point'])}")
    print(f"   Природа:    {global_min['type']} точка\n")

    print("="*45)
    print("ПОЛНЫЙ СПИСОК КАНДИДАТОВ")
    print("="*45)

    for i, data in enumerate(valid_points, 1):
        f_val = data['f_val']
        coords_str = format_coords(data['point'])
        p_type = data['type']

        if data == global_max:
            status = "Глобальный максимум"
        elif data == global_min:
            status = "Глобальный минимум"
        else:
            status = "Стационарная (локальный экстр. или седло)"

        print(f"{i}. {status}")
        print(f"   f = {f_val} | Точка: ({coords_str}) | Тип: {p_type}\n")


def main():
  n, f, m, gs = parse_data(input)
  nonreg = find_nonreg_solves(n, f, m, gs)
  reg = find_reg_solves(n, f, m, gs)
  recognize_type(nonreg + reg)

main()


[МАКСИМУМ] f = 11.0
   Координаты: x = 3, y = 3, z = 2
   Природа:    regular точка

[МИНИМУМ]  f = -11.0
   Координаты: x = -3, y = -3, z = -2
   Природа:    regular точка

ПОЛНЫЙ СПИСОК КАНДИДАТОВ
1. Глобальный максимум
   f = 11.0 | Точка: (x = 3, y = 3, z = 2) | Тип: regular

2. Глобальный минимум
   f = -11.0 | Точка: (x = -3, y = -3, z = -2) | Тип: regular

